<a href="https://colab.research.google.com/github/Anoshafatima131/ML-Project/blob/main/Methodology_Improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
wisam1985_advanced_iot_agriculture_2024_path = kagglehub.dataset_download('wisam1985/advanced-iot-agriculture-2024')
anoshafatima131_advanced_iot_agriculture_path = kagglehub.notebook_output_download('anoshafatima131/advanced-iot-agriculture')

print('Data source import complete.')


In [ ]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [ ]:
# ===============================
# 2. LOAD DATA
# ===============================
df = pd.read_csv('/kaggle/input/notebooks/anoshafatima131/advanced-iot-agriculture/preprocessed_iot_agriculture_dataset.csv')

df.head()

In [ ]:
# ===============================
# 3. FEATURES & TARGET
# ===============================
X = df.drop("Class", axis=1)
y = df["Class"]

# Convert target into categories
y = pd.cut(y, bins=3, labels=[0,1,2])

In [ ]:
# ===============================
# 4. TRAIN TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ===============================
# 5. FEATURE SCALING (NEW IMPROVEMENT)
# ===============================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# ===============================
# 6. MODELS
# ===============================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC()
}

In [ ]:
# ===============================
# 7. TRAIN + EVALUATE FUNCTION
# ===============================
def evaluate(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(name)
    plt.show()

    return accuracy_score(y_test, y_pred)

In [ ]:
# ===============================
# 8. MODEL COMPARISON
# ===============================
results = {}

for name, model in models.items():
    acc = evaluate(model, X_train, X_test, y_train, y_test, name)
    results[name] = acc

results_df = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
print(results_df)

sns.barplot(x="Model", y="Accuracy", data=results_df)
plt.title("Model Comparison")
plt.show()

**CROSS VALIDATION (BIG IMPROVEMENT)**

In [ ]:
print("\nCross Validation Scores:")

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"{name}: Mean Accuracy = {scores.mean():.4f}")

**HYPERPARAMETER TUNING**

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid,
                    cv=3,
                    n_jobs=-1)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

In [ ]:
# ===============================
# 11. FINAL MODEL
# ===============================
final_model = grid.best_estimator_

y_pred_final = final_model.predict(X_test)

print("\nFinal Model Accuracy:", accuracy_score(y_test, y_pred_final))

In [ ]:
# ===============================
# 12. FINAL PREDICTION
# ===============================
sample = X_test[0].reshape(1, -1)

prediction = final_model.predict(sample)

print("Predicted Plant Growth Class:", prediction)

### Improvements Implemented

- Applied feature scaling using StandardScaler
- Used cross-validation for better model reliability
- Performed hyperparameter tuning using GridSearchCV
- Selected best model based on optimized performance
- Reduced chances of overfitting